# Coral Detection — Model Comparison Report

Loads the three **trained** models and measures them **on the same chosen evaluation split with the same metric**, so the "best performing algorithm" conclusion is defensible.

**Models:** YOLOv8n · Faster R-CNN (ResNet50-FPN) · EfficientDet-D0

**Everything below is measured, not assumed:**
- **Accuracy** — COCO **mAP** (torchmetrics), the standard object-detection metric, computed on the test split for all three.
- **Speed** — real inference time (warm-up + timed runs), reported in ms and FPS.
- **Model size** — actual `.pt` file size on disk.

> ⚠️ This notebook was rebuilt. The previous version faked its numbers: inference speed came from `time.sleep()`, and the accuracies were hard-coded (`0.85 / 0.92 / 0.88`). None of it measured the real models.
Validation is the default evaluation split. Before a long run, use one epoch as a smoke test, inspect boxes and losses, then start a fresh full experiment. Old five-class checkpoints cannot be used in the new comparison.


## Evaluation metrics & mathematical formulas

Object detection is **not** scored with plain classification "accuracy," because each image contains many objects that must be both **located** and **classified**. The field-standard metric is **mean Average Precision (mAP)**. It is built up from the following definitions.

**1. Intersection over Union (IoU)** — how well a predicted box $B_p$ overlaps a ground-truth box $B_{gt}$:

$$\mathrm{IoU} = \frac{\text{area}(B_p \cap B_{gt})}{\text{area}(B_p \cup B_{gt})}$$

A prediction is a **True Positive (TP)** if it has the correct class **and** $\mathrm{IoU} \ge t$ (threshold, e.g. 0.5). Otherwise it is a **False Positive (FP)**. A ground-truth object with no matching prediction is a **False Negative (FN)**.

**2. Precision and Recall:**

$$\text{Precision} = \frac{TP}{TP + FP} \qquad\qquad \text{Recall} = \frac{TP}{TP + FN}$$

**3. F1-score** (harmonic mean of precision and recall):

$$F_1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text{Precision} + \text{Recall}}$$

**4. Average Precision (AP)** — the area under the precision–recall curve for one class, i.e. precision $p(r)$ integrated over recall $r$:

$$\text{AP} = \int_{0}^{1} p(r)\,dr \;\approx\; \sum_{n}\left(r_{n+1}-r_{n}\right)\,p_{\text{interp}}(r_{n+1})$$

**5. mean Average Precision (mAP)** — AP averaged over all $C$ classes:

$$\text{mAP} = \frac{1}{C}\sum_{i=1}^{C}\text{AP}_i$$

Two conventions are reported (as in the COCO benchmark):
- **mAP@0.5** — TP threshold fixed at $\mathrm{IoU}=0.5$ (the "PASCAL VOC" convention).
- **mAP@0.5:0.95** — averaged over ten IoU thresholds $0.50, 0.55, \dots, 0.95$; the stricter, primary COCO metric.

**Speed** is reported as inference latency (ms/image) and throughput $\text{FPS} = 1000 / \text{latency(ms)}$.

> **Reading the results:** the `quadrat_area` box is large and easy to detect, so it earns a very high AP and pulls the overall mAP up for every model. When you argue the "best" algorithm to the panel, compare the **per-class AP on the seven benthic classes** — that reflects the genuinely hard task — alongside the overall mAP, speed, and size trade-off.

## Setup: Install Dependencies

In [ ]:
!pip install -q ultralytics effdet timm torchmetrics pycocotools roboflow pyyaml tqdm matplotlib pandas

## Configure the reviewed dataset and checkpoint storage

Use the same version as all three training runs. Validation is the default; change EVAL_SPLIT to test only for the final evaluation.

In [ ]:
MODEL_NAME="comparison"
from pathlib import Path
from getpass import getpass
from datetime import datetime
import os, json, random, hashlib, importlib.metadata
import numpy as np
import torch
from roboflow import Roboflow

# Fill these from the newly generated Roboflow version. Never paste a key here.
WORKSPACE = ""
PROJECT = ""
DATASET_VERSION = None
EXPECTED_CLASSES = {"Hard Coral", "Soft Coral", "quadrat_area"}
SEED = 42
USE_DRIVE = True
EVAL_SPLIT = "valid"  # Use test only for the final, frozen evaluation.
assert WORKSPACE and PROJECT and isinstance(DATASET_VERSION, int), "Enter workspace, project and version first."
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda": print("GPU:", torch.cuda.get_device_name(0))
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_ROOT = Path('/content/drive/MyDrive/CoralSense')
else:
    OUTPUT_ROOT = Path.cwd() / 'coralsense_outputs'
CHECKPOINT_DIR = OUTPUT_ROOT / 'checkpoints'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR = OUTPUT_ROOT / 'runs' / (datetime.now().strftime('%Y%m%d-%H%M%S-%f') + '-' + MODEL_NAME)
RUN_DIR.mkdir(parents=True)
ROBOFLOW_API_KEY = getpass("Roboflow API key: ")
dataset = Roboflow(api_key=ROBOFLOW_API_KEY).workspace(WORKSPACE).project(PROJECT).version(DATASET_VERSION).download('yolov8')
del ROBOFLOW_API_KEY
dataset_path = Path(dataset.location)

## Audit the dataset and load all three matching checkpoints

In [ ]:
import yaml
from PIL import Image, ImageOps
import torchvision.transforms.functional as TF

with (dataset_path / 'data.yaml').open() as handle:
    data = yaml.safe_load(handle)
names = data['names']
class_names = [names[k] for k in sorted(names, key=int)] if isinstance(names, dict) else list(names)
assert len(class_names) == 3 and set(class_names) == EXPECTED_CLASSES, f"Unexpected classes: {class_names}"
data['names'] = class_names
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}
def image_paths(directory):
    return sorted(p for p in Path(directory).iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS)

def find_split(dp, split_names):
    for name in split_names:
        aliases = ['valid', 'val'] if name in ('valid', 'val') else [name]
        for split in aliases:
            for images, labels in [(dp / split / 'images', dp / split / 'labels'),
                                   (dp / 'images' / split, dp / 'labels' / split)]:
                if images.is_dir() and labels.is_dir() and image_paths(images): return images, labels
    raise FileNotFoundError(f"Missing/empty split: {split_names}. No silent evaluation fallback.")

def read_annotations(path, label_dir, width, height, offset=0):
    boxes, labels = [], []
    label_path = Path(label_dir) / (Path(path).stem + '.txt')
    if not label_path.exists():
        raise FileNotFoundError(f"Missing label: {label_path.name}. Use an empty file only for reviewed negative images.")
    for line in label_path.read_text().splitlines():
        if not line.strip(): continue
        fields = line.split()
        if len(fields) != 5: raise ValueError(f"Expected detection boxes, not polygons: {label_path.name}")
        cid = int(fields[0]); cx, cy, w, h = map(float, fields[1:])
        if not (0 <= cid < len(class_names) and all(np.isfinite([cx,cy,w,h])) and
                0 <= cx <= 1 and 0 <= cy <= 1 and 0 < w <= 1 and 0 < h <= 1):
            raise ValueError(f"Invalid class/box in {label_path.name}")
        box = [max(0., (cx-w/2)*width), max(0., (cy-h/2)*height),
               min(float(width), (cx+w/2)*width), min(float(height), (cy+h/2)*height)]
        if box[2] <= box[0] or box[3] <= box[1]: raise ValueError(f"Degenerate box: {label_path.name}")
        boxes.append(box); labels.append(cid + offset)
    return {'boxes': torch.tensor(boxes, dtype=torch.float32).reshape(-1,4),
            'labels': torch.tensor(labels, dtype=torch.int64)}

split_dirs = {s: find_split(dataset_path, [s]) for s in ('train','valid','test')}
manifest = {}; seen_hashes = {}; fingerprint = hashlib.sha256()
for split, (images_dir, labels_dir) in split_dirs.items():
    counts = np.zeros(len(class_names), dtype=int); rows = []
    for path in image_paths(images_dir):
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        if digest in seen_hashes and seen_hashes[digest] != split:
            raise ValueError('Identical image found across splits: ' + path.name)
        seen_hashes[digest] = split
        with Image.open(path) as img: target = read_annotations(path, labels_dir, *img.size)
        for cid in target['labels'].tolist(): counts[cid] += 1
        label_hash = hashlib.sha256((labels_dir / (path.stem+'.txt')).read_bytes()).hexdigest()
        rows.append({'image_sha256': digest, 'label_sha256': label_hash})
    if any(counts == 0): raise ValueError(f"{split} has no examples of: {[class_names[i] for i in np.where(counts==0)[0]]}")
    # Filenames differ across exports; content-based fingerprints remain comparable.
    rows.sort(key=lambda r: (r['image_sha256'],r['label_sha256']))
    manifest[split] = {'images':len(rows),'instances':counts.tolist(),'content':rows}
    print(split, 'images:',len(rows), 'instances:',dict(zip(class_names,counts.tolist())))
fingerprint.update(json.dumps({'classes':class_names,'splits':manifest},sort_keys=True).encode())
DATASET_FINGERPRINT = fingerprint.hexdigest()
META = {'workspace':WORKSPACE,'project':PROJECT,'version':DATASET_VERSION,
        'class_names':class_names,'dataset_fingerprint':DATASET_FINGERPRINT,'seed':SEED,
        'model':MODEL_NAME,'hardware':torch.cuda.get_device_name(0) if device.type=='cuda' else 'cpu',
        'augmentation':'Roboflow version transformations only; additional random training augmentation disabled'}
(RUN_DIR/'dataset_manifest.json').write_text(json.dumps(manifest,indent=2))
(RUN_DIR/'run_metadata.json').write_text(json.dumps(META,indent=2))
versions = {}
for name in ('torch','torchvision','roboflow','numpy','Pillow','ultralytics','torchmetrics','effdet','timm','pycocotools'):
    try: versions[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: pass
(RUN_DIR/'package_versions.json').write_text(json.dumps(versions,indent=2))
print('Classes:',class_names,'Output:',RUN_DIR)
print('Exact duplicates checked. Related photos must also have been grouped by survey/session before export.')

def letterbox(img, size):
    w,h=img.size; ratio=min(size/w,size/h)
    nw,nh=max(1,round(w*ratio)),max(1,round(h*ratio))
    left,top=(size-nw)//2,(size-nh)//2
    canvas=Image.new('RGB',(size,size),(114,114,114))
    canvas.paste(img.resize((nw,nh),Image.Resampling.BILINEAR),(left,top))
    return canvas,(nw/w,nh/h,left,top)

from ultralytics import YOLO
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from effdet import get_efficientdet_config, EfficientDet, DetBenchPredict
from effdet.efficientdet import HeadNet
def check_meta(meta):
    assert meta['class_names']==class_names,'Checkpoint class order differs from dataset'
    assert meta['dataset_fingerprint']==DATASET_FINGERPRINT,'Checkpoint trained on different dataset/splits'
    assert (meta['workspace'],meta['project'],meta['version'])==(WORKSPACE,PROJECT,DATASET_VERSION),'Dataset version mismatch'
yolo_path=CHECKPOINT_DIR/'yolov8_coral_best.pt'
frcnn_path=CHECKPOINT_DIR/'faster_rcnn_coral_best.pt'
edet_path=CHECKPOINT_DIR/'efficientdet_coral_best.pt'
for p in (yolo_path,frcnn_path,edet_path):assert p.is_file(),f'Missing trained checkpoint: {p.name}'
check_meta(json.loads(yolo_path.with_suffix('.metadata.json').read_text()))
yolo_model=YOLO(str(yolo_path))
assert [yolo_model.names[i] for i in sorted(yolo_model.names)]==class_names
# Only load checkpoints you trained/trust. These contain optimizer metadata.
ckpt=torch.load(frcnn_path,map_location='cpu',weights_only=False);check_meta(ckpt['metadata'])
frcnn_model=fasterrcnn_resnet50_fpn(weights=None,weights_backbone=None,box_score_thresh=.001,box_detections_per_img=100)
frcnn_model.roi_heads.box_predictor=FastRCNNPredictor(frcnn_model.roi_heads.box_predictor.cls_score.in_features,ckpt['num_classes'])
frcnn_model.load_state_dict(ckpt['model_state_dict']);frcnn_model.to(device).eval()
eckpt=torch.load(edet_path,map_location='cpu',weights_only=False);check_meta(eckpt['metadata'])
EDET_IMG=eckpt['image_size'];config=get_efficientdet_config(eckpt['architecture'])
config.num_classes=eckpt['num_classes'];config.image_size=(EDET_IMG,EDET_IMG)
enet=EfficientDet(config,pretrained_backbone=False);enet.class_net=HeadNet(config,num_outputs=eckpt['num_classes'])
enet.load_state_dict(eckpt['model_state_dict']);edet_bench=DetBenchPredict(enet).to(device).eval()
test_img_dir,test_lbl_dir=split_dirs[EVAL_SPLIT];eval_split=EVAL_SPLIT
test_images=image_paths(test_img_dir)
yolo_size=yolo_path.stat().st_size/1024**2;frcnn_size=frcnn_path.stat().st_size/1024**2;edet_size=edet_path.stat().st_size/1024**2
print('All checkpoints match. Evaluating',eval_split,len(test_images),'images.')

In [ ]:
## Dataset Class Distribution

#Counts of annotated **instances (bounding boxes)** and **images** per class across the
#train / val / test splits. This answers two panel comments:

##1. *"The distribution of the images per class should be discussed in the documentation."* — the table and chart below are the figures to put in your Dataset chapter.
##2. *"Consider the distribution of datasets in the comparative analysis of the algorithm."* — reference these counts when interpreting the per-class AP results later (e.g. a class with few instances explaining a low AP).

#> Note: If Roboflow augmentation was enabled, only training counts are increased; inspect the recorded version settings — the **relative proportions between classes** are what reveal imbalance. `quadrat_area` appears about once per image, so its instance count ≈ its image count.
# ── Dataset class distribution (instances + images per class, per split) ────
# Self-contained: only needs `dataset_path`, `class_names`, and `find_split`
# from the dataset-loading cell above.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import defaultdict

SPLITS = ['train', 'val', 'test']
inst_counts = {s: defaultdict(int) for s in SPLITS}   # bounding boxes per class
img_counts = {s: defaultdict(int) for s in SPLITS}    # images containing the class

for s in SPLITS:
    _, lbl_dir = find_split(dataset_path, [s, 'valid'] if s == 'val' else [s])
    if lbl_dir is None:
        continue
    for image_path in image_paths(find_split(dataset_path, [s])[0]):
        lbl = lbl_dir / (image_path.stem + '.txt')
        present = set()
        for line in open(lbl):
            p = line.split()
            if len(p) < 5:
                continue
            cid = int(p[0])
            if 0 <= cid < len(class_names):
                inst_counts[s][cid] += 1
                present.add(cid)
        for cid in present:
            img_counts[s][cid] += 1

# ---- Table 1: instances (bounding boxes) per class ----
inst_rows = []
for cid, name in enumerate(class_names):
    tr, va, te = inst_counts['train'][cid], inst_counts['val'][cid], inst_counts['test'][cid]
    inst_rows.append({'Class': name, 'Train': tr, 'Val': va, 'Test': te, 'Total': tr + va + te})
inst_df = pd.DataFrame(inst_rows)
total_inst = inst_df['Total'].sum()
inst_df['% of all'] = (inst_df['Total'] / total_inst * 100).round(1) if total_inst else 0

# ---- Table 2: images per class ----
img_rows = []
for cid, name in enumerate(class_names):
    tr, va, te = img_counts['train'][cid], img_counts['val'][cid], img_counts['test'][cid]
    img_rows.append({'Class': name, 'Train': tr, 'Val': va, 'Test': te, 'Total': tr + va + te})
img_df = pd.DataFrame(img_rows)

print("=" * 74)
print("CLASS DISTRIBUTION — annotated INSTANCES (bounding boxes)")
print("=" * 74)
print(inst_df.to_string(index=False))
print(f"\nTotal instances: {total_inst}")

print("\n" + "=" * 74)
print("CLASS DISTRIBUTION — IMAGES containing each class")
print("=" * 74)
print(img_df.to_string(index=False))

# ---- Imbalance ratio (over benthic classes, ignoring quadrat_area) ----
benthic_totals = inst_df[inst_df['Class'] != 'quadrat_area']['Total']
benthic_totals = benthic_totals[benthic_totals > 0]
if len(benthic_totals):
    imb = benthic_totals.max() / benthic_totals.min()
    print(f"\nBenthic class imbalance ratio (largest / smallest): {imb:.1f}×")
    print("→ Discuss this when interpreting per-class AP: rare classes tend to score lower.")

# ---- Save for the documentation ----
inst_df.to_csv(RUN_DIR / 'class_distribution_instances.csv', index=False)
img_df.to_csv(RUN_DIR / 'class_distribution_images.csv', index=False)
print("\n✓ Saved: class_distribution_instances.csv, class_distribution_images.csv")

# ---- Chart: stacked bar of instances per class, coloured by split ----
fig, ax = plt.subplots(figsize=(11, 5))
bottom = np.zeros(len(class_names))
split_colors = {'Train': '#2a78d6', 'Val': '#eda100', 'Test': '#1baf7a'}
for split in ['Train', 'Val', 'Test']:
    vals = inst_df[split].values
    ax.bar(inst_df['Class'], vals, bottom=bottom, label=split,
           color=split_colors[split], edgecolor='black', linewidth=0.4)
    bottom += vals
ax.set_ylabel('Annotated instances', fontweight='bold')
ax.set_title('Dataset Class Distribution (instances per class, by split)', fontweight='bold')
ax.legend(title='Split')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig(RUN_DIR / 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Saved: class_distribution.png")

## Benchmark: Accuracy (mAP) + Speed + Size — all measured

The next cell runs every loaded model over the test split, computing COCO mAP with `torchmetrics`, timing real inference (with GPU warm-up), and recording each checkpoint's file size. Results are stored in the `results` dict used by the charts and report below.

In [ ]:
import time
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from tqdm.auto import tqdm
import torchvision.transforms as T
to_tensor=T.ToTensor()
edet_norm=T.Normalize([.485,.456,.406],[.229,.224,.225])
@torch.inference_mode()
def predict(name,path,conf=.001):
    # All paths include image decoding, preprocessing, transfer and postprocessing.
    with Image.open(path) as original:img=original.convert('RGB')
    W,H=img.size
    if name=='YOLOv8n':
        r=yolo_model.predict(img,device=0 if device.type=='cuda' else 'cpu',conf=conf,imgsz=640,max_det=100,verbose=False)[0]
        return {'boxes':r.boxes.xyxy.cpu(),'scores':r.boxes.conf.cpu(),'labels':r.boxes.cls.cpu().long()}
    if name=='Faster R-CNN':
        p=frcnn_model([to_tensor(img).to(device)])[0];keep=p['scores']>=conf
        return {'boxes':p['boxes'][keep].cpu(),'scores':p['scores'][keep].cpu(),'labels':p['labels'][keep].cpu()-1}
    canvas,(sx,sy,left,top)=letterbox(img,EDET_IMG)
    rows=edet_bench(edet_norm(to_tensor(canvas)).unsqueeze(0).to(device))[0].cpu()
    rows=rows[rows[:,4]>=conf];boxes=rows[:,:4].clone()
    boxes[:,[0,2]]=(boxes[:,[0,2]]-left)/sx;boxes[:,[1,3]]=(boxes[:,[1,3]]-top)/sy
    boxes[:,[0,2]]=boxes[:,[0,2]].clamp(0,W);boxes[:,[1,3]]=boxes[:,[1,3]].clamp(0,H)
    keep=(boxes[:,2]>boxes[:,0])&(boxes[:,3]>boxes[:,1])
    return {'boxes':boxes[keep],'scores':rows[keep,4],'labels':rows[keep,5].long()-1}
results={};per_class={}
for name,size in [('YOLOv8n',yolo_size),('Faster R-CNN',frcnn_size),('EfficientDet-D0',edet_size)]:
    metric=MeanAveragePrecision(box_format='xyxy',class_metrics=True)
    for path in tqdm(test_images,desc=name+' AP'):
        with Image.open(path) as image:truth=read_annotations(path,test_lbl_dir,*image.size)
        metric.update([predict(name,path)],[truth])
    res=metric.compute();per_class[name]=res
    timing_images=test_images[:min(5,len(test_images))]
    for _ in range(3):predict(name,timing_images[0],conf=.25)
    times=[]
    for _ in range(3):
        for path in timing_images:
            if device.type=='cuda':torch.cuda.synchronize()
            start=time.perf_counter();predict(name,path,conf=.25)
            if device.type=='cuda':torch.cuda.synchronize()
            times.append((time.perf_counter()-start)*1000)
    ms=float(np.mean(times))
    results[name]={'map50':float(res['map_50']),'map':float(res['map']),'ms':ms,'fps':1000/ms,'size':size}
    print(name,results[name])
(RUN_DIR/'benchmark.json').write_text(json.dumps({'metadata':META,'split':EVAL_SPLIT,'results':results},indent=2))

## Visualization: Speed vs Size vs Accuracy

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = list(results.keys())
fps = [results[m]['fps'] for m in models]
sizes = [results[m]['size'] for m in models]
map50 = [results[m]['map50'] for m in models]
map5095 = [results[m]['map'] for m in models]
colors = ['#4CAF50', '#FF9800', '#2196F3'][:len(models)]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Coral Detection — Model Comparison (evaluation split, measured)', fontsize=16, fontweight='bold')

def bar(ax, vals, title, ylabel, fmt):
    ax.bar(models, vals, color=colors, alpha=0.85, edgecolor='black', linewidth=1.5)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=11, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    for i, v in enumerate(vals):
        ax.text(i, v, fmt.format(v), ha='center', va='bottom', fontweight='bold')

bar(axes[0, 0], fps, 'Inference Speed', 'FPS', '{:.1f}')
bar(axes[0, 1], sizes, 'Model Size', 'MB', '{:.1f}')

# Accuracy: mAP@0.5 and mAP@0.5:0.95 side by side
ax = axes[1, 0]
x = np.arange(len(models)); w = 0.38
ax.bar(x - w / 2, map50, w, label='mAP@0.5', color='#1c6d7d', edgecolor='black')
ax.bar(x + w / 2, map5095, w, label='mAP@0.5:0.95', color='#8fb9c4', edgecolor='black')
ax.set_xticks(x); ax.set_xticklabels(models)
ax.set_ylabel('mAP', fontweight='bold'); ax.set_ylim(0, 1)
ax.set_title('Detection Accuracy (mAP)', fontsize=12, fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
for i in range(len(models)):
    ax.text(i - w / 2, map50[i], f'{map50[i]:.2f}', ha='center', va='bottom', fontsize=9)
    ax.text(i + w / 2, map5095[i], f'{map5095[i]:.2f}', ha='center', va='bottom', fontsize=9)

# Speed vs accuracy trade-off
ax = axes[1, 1]
ax.scatter(fps, map5095, s=[sz * 4 for sz in sizes], c=colors, alpha=0.7, edgecolor='black')
for i, m in enumerate(models):
    ax.annotate(m, (fps[i], map5095[i]), textcoords="offset points", xytext=(8, 5), fontweight='bold')
ax.set_xlabel('Speed (FPS) →', fontweight='bold')
ax.set_ylabel('Accuracy (mAP@0.5:0.95) →', fontweight='bold')
ax.set_title('Speed vs Accuracy (bubble = model size)', fontsize=12, fontweight='bold')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print("✓ Comparison charts generated from measured values.")

## Generate Comparison Report

In [ ]:
import pandas as pd

# ── Main comparison table (all values measured) ─────────────────────────────
rows = []
for m in models:
    r = results[m]
    rows.append({
        'Model': m,
        'mAP@0.5': round(r['map50'], 3),
        'mAP@0.5:0.95': round(r['map'], 3),
        'Inference (ms)': round(r['ms'], 1),
        'FPS': round(r['fps'], 1),
        'Size (MB)': round(r['size'], 1),
        'Real-Time (>=15 FPS)': 'Yes' if r['fps'] >= 15 else 'No',
    })
df = pd.DataFrame(rows)

print("=" * 90)
print(f"MODEL COMPARISON — measured on the {eval_split} split ({len(test_images)} images)")
print("=" * 90)
print(df.to_string(index=False))

# ── Per-class AP, and a benthic-only mean (quadrat_area excluded) ───────────
def per_class_ap(model_name):
    """Return {class_name: AP@0.5:0.95} for a model."""
    res = per_class[model_name]
    if 'map_per_class' not in res or res['classes'].numel() == 0:
        return {}
    offset = 0  # predict() normalizes every model to dataset IDs
    out = {}
    pc = res['map_per_class']
    for i, c in enumerate(res['classes'].tolist()):
        idx = c - offset
        if 0 <= idx < len(class_names):
            ap = pc[i].item() if pc.numel() > 1 else pc.item()
            out[class_names[idx]] = ap
    return out

benthic = [c for c in class_names if c != 'quadrat_area']
print("\n" + "=" * 90)
print("PER-CLASS AP@0.5:0.95  (benthic classes are the genuinely hard task)")
print("=" * 90)
pc_table = {}
for m in models:
    aps = per_class_ap(m)
    pc_table[m] = aps
    coral_vals = [aps[c] for c in benthic if c in aps and aps[c] >= 0]
    results[m]['benthic_map'] = float(np.mean(coral_vals)) if coral_vals else float('nan')

pc_df = pd.DataFrame(pc_table).round(3)
print(pc_df.to_string())
print("\nBenthic-only mean AP@0.5:0.95 (excludes quadrat_area):")
for m in models:
    print(f"  {m:<16}: {results[m]['benthic_map']:.3f}")

# ── Data-driven recommendation ──────────────────────────────────────────────
best_overall = max(models, key=lambda m: results[m]['map'])
best_benthic = max(models, key=lambda m: results[m]['benthic_map'])
fastest = max(models, key=lambda m: results[m]['fps'])
smallest = min(models, key=lambda m: results[m]['size'])

print("\n" + "=" * 90)
print("CONCLUSION (from measured results)")
print("=" * 90)
print(f"• Highest overall mAP@0.5:0.95 : {best_overall} ({results[best_overall]['map']:.3f})")
print(f"• Highest benthic-only mAP     : {best_benthic} ({results[best_benthic]['benthic_map']:.3f})")
print(f"• Fastest                      : {fastest} ({results[fastest]['fps']:.1f} FPS)")
print(f"• Smallest                     : {smallest} ({results[smallest]['size']:.1f} MB)")
print("\nFor the panel: state the best-performing algorithm as the one with the highest")
print("mAP (cite both the overall and the benthic-only figure), and discuss the")
print("speed/size trade-off for real-world field deployment.")

## Test Detection Results Comparison

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
sample=test_images[0]
fig,axes=plt.subplots(1,3,figsize=(21,7))
for ax,name in zip(axes,results):
    with Image.open(sample) as original:img=original.convert('RGB')
    prediction=predict(name,sample,conf=.3);ax.imshow(img);ax.set_title(name)
    for box,score,cid in zip(prediction['boxes'],prediction['scores'],prediction['labels']):
        x1,y1,x2,y2=box.tolist()
        ax.add_patch(patches.Rectangle((x1,y1),x2-x1,y2-y1,fill=False,edgecolor='lime'))
        ax.text(x1,y1,f'{class_names[int(cid)]} {float(score):.2f}',color='lime')
plt.show()

## Export Results

In [ ]:
df.to_csv(RUN_DIR/'model_comparison_report.csv',index=False)
pc_df.to_csv(RUN_DIR/'per_class_ap.csv')
print('Reports and metadata:',RUN_DIR)
print('Select settings using validation. Report final test results only after selection is frozen.')